# Minimal combat PPO training

Same training loop as ``TrainCombat.ipynb``, but the policy is ``MinimalCombatPPOAgent`` (``agent.minimal_model``): hand and deck cards (rank + suit, shared embedding plus a learned deck offset), run scalars (plays/discards/hand size and scaled scores), and the 12 poker-hand level rows, with hand→deck then hand→levels+run cross-attention. It ignores jokers, boss, and card modifiers that still appear in the flat observation dict.

**Setup:** The next cell sets ``REPO_ROOT`` (your clone), adds imports on ``sys.path``, ``SNAPSHOT_JSON_DIR`` (folder of ``*.json`` snapshots), and ``VEC_BACKEND`` (``"async"`` or ``"sync"``).

**Checkpoints:** Written under ``<REPO_ROOT>/checkpoints/`` with the ``minimal_combat_ppo_iter_*.pt`` filename prefix. These are **not** compatible with checkpoints from the full-model notebook (``combat_ppo_iter_*.pt`` / ``CombatPPOAgent``). In the training-setup cell, set ``RESUME_CHECKPOINT`` to a minimal run’s ``.pt`` path or ``None``. Use ``RESUME_MODEL_WEIGHTS_ONLY = True`` to load only ``model_state_dict`` (keep this notebook’s ``PPOConfig``, iteration 1, fresh optimizer). Checkpoints are loaded with ``weights_only=False`` because they store ``PPOConfig`` and other non-tensor data.

**Logging:** Per-iteration line (last 50 episodes): ``lr``, ``c_entropy``, ``clip_eps``, ``reward``, ``len``, ``win%``, ``pg``, ``vf``, ``ent``, ``episodes``; then **P(select)** per hand slot for env 0 and **std** (same hyperparams repeated); checkpoint saves repeat them too.


In [1]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass

# Colab: Drive path to this repo. Local: path to your checkout (absolute).
REPO_ROOT = Path("/content/drive/MyDrive/balatro-agent-project")

os.chdir(REPO_ROOT)
for p in (str(REPO_ROOT), str(REPO_ROOT / "balatro_lite_gym")):
    if p not in sys.path:
        sys.path.insert(0, p)

SNAPSHOT_JSON_DIR = REPO_ROOT / "snapshots"
VEC_BACKEND = "async"  # "sync" if vector env workers misbehave
if VEC_BACKEND not in ("async", "sync"):
    raise ValueError(f"VEC_BACKEND must be 'async' or 'sync', got {VEC_BACKEND!r}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import torch
from torch.distributions import Categorical

from env.lite_combat_env import (
    VecRolloutBuffer,
    compute_gae_vectorized,
    dict_to_tensors,
    make_vec,
)
from agent.minimal_model import MinimalCombatPPOAgent
from agent.ppo import get_card_mask, mask_logits, ppo_update
from agent.ppo_config import PPOConfig
from env.snapshot_io import load_snapshot_pool_from_json_dir


In [3]:
snapshot_pool = load_snapshot_pool_from_json_dir(SNAPSHOT_JSON_DIR)
print("pool size:", len(snapshot_pool))


pool size: 101


In [4]:
RESUME_CHECKPOINT = REPO_ROOT / "checkpoints" / "minimal_combat_ppo_iter_1000.pt"  # str path or None, e.g. REPO_ROOT / "checkpoints" / "minimal_combat_ppo_iter_100.pt"
RESUME_MODEL_WEIGHTS_ONLY = True

cfg = PPOConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
start_iteration = 1

if isinstance(RESUME_CHECKPOINT, str) and not RESUME_CHECKPOINT.strip():
    RESUME_CHECKPOINT = None
elif RESUME_CHECKPOINT is not None:
    RESUME_CHECKPOINT = str(RESUME_CHECKPOINT)
# True: load only model weights; keep cfg / start_iteration above; fresh optimizer (architecture must match).

CKPT_EVERY = 100
ckpt = None

if RESUME_CHECKPOINT:
    try:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
    if RESUME_MODEL_WEIGHTS_ONLY:
        print(
            f"Loaded model weights from {RESUME_CHECKPOINT!r} "
            f"(cfg/optimizer/iteration unchanged; training starts at iteration {start_iteration})"
        )
    else:
        cfg = ckpt["config"]
        start_iteration = int(ckpt["iteration"]) + 1
        print(f"Resumed from {RESUME_CHECKPOINT!r} — next iteration {start_iteration}")

assert (cfg.rollout_steps * cfg.num_envs) % cfg.num_minibatches == 0, (
    f"rollout_steps * num_envs must divide num_minibatches; got "
    f"{cfg.rollout_steps} * {cfg.num_envs} vs {cfg.num_minibatches}"
)

agent = MinimalCombatPPOAgent(
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    dim_ff=cfg.dim_ff,
    dropout=cfg.dropout,
).to(device)
optimizer = torch.optim.Adam(agent.parameters(), lr=cfg.lr, eps=1e-5)

if ckpt is not None:
    agent.load_state_dict(ckpt["model_state_dict"])
    if not RESUME_MODEL_WEIGHTS_ONLY:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

if cfg.use_lr_scheduler:
    total_iters = max(1, cfg.max_iterations)
    lr_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=cfg.lr_min_ratio,
        total_iters=total_iters,
    )

    if start_iteration > 1:
        for _ in range(start_iteration - 1):
            lr_scheduler.step()
else:
    lr_scheduler = None

vec = make_vec(
    snapshot_pool,
    n=cfg.num_envs,
    base_seed=0,
    backend=VEC_BACKEND,
)
buffer = VecRolloutBuffer(cfg.rollout_steps, cfg.num_envs, device)

print("device:", device, "| vector backend:", VEC_BACKEND)


Loaded model weights from '/content/drive/MyDrive/balatro-agent-project/checkpoints/minimal_combat_ppo_iter_1000.pt' (cfg/optimizer/iteration unchanged; training starts at iteration 1)
device: cuda | vector backend: async


In [5]:
def _per_env_combat_won(infos, num_envs: int) -> np.ndarray:
    if not isinstance(infos, dict) or "combat_won" not in infos:
        return np.zeros(num_envs, dtype=bool)
    cw = np.asarray(infos["combat_won"])
    if cw.dtype == object:
        row = [bool(x) for x in cw.tolist()[:num_envs]]
        return np.pad(row, (0, max(0, num_envs - len(row))), constant_values=False)
    flat = cw.astype(bool).reshape(-1)[:num_envs]
    if flat.size < num_envs:
        out = np.zeros(num_envs, dtype=bool)
        out[: flat.size] = flat
        return out
    return flat


def _wins_from_snapshot_fallback(infos, num_envs: int, dones_np: np.ndarray) -> np.ndarray:
    out = np.zeros(num_envs, dtype=bool)
    if not isinstance(infos, dict) or "snapshot" not in infos or not dones_np.any():
        return out
    snaps = infos["snapshot"]
    if isinstance(snaps, (list, tuple)):
        seq = list(snaps)
    else:
        seq = [snaps] * num_envs
    for i in np.where(dones_np)[0]:
        if i < len(seq):
            s = seq[i]
            out[i] = bool(s.current_score >= s.target_score)
    return out


def _episode_wins(infos, num_envs: int, dones_np: np.ndarray) -> np.ndarray:
    w = _per_env_combat_won(infos, num_envs)
    if isinstance(infos, dict) and "combat_won" not in infos:
        w = _wins_from_snapshot_fallback(infos, num_envs, dones_np)
    return w


def _per_env_pbrs_gamma(infos, num_envs: int) -> np.ndarray:
    if not isinstance(infos, dict) or "pbrs_gamma" not in infos:
        raise KeyError("vector step infos must include pbrs_gamma (BalatroEnv)")
    g = np.asarray(infos["pbrs_gamma"])
    if g.dtype == object:
        row = [float(x) for x in g.tolist()[:num_envs]]
        out = np.zeros(num_envs, dtype=np.float32)
        n = min(len(row), num_envs)
        out[:n] = np.asarray(row[:n], dtype=np.float32)
        return out
    flat = g.astype(np.float32).reshape(-1)[:num_envs]
    if flat.size < num_envs:
        out = np.zeros(num_envs, dtype=np.float32)
        out[: flat.size] = flat
        return out
    return flat


In [6]:
class LinearScheduler:
    def __init__(self, start_val: float, end_val: float, total_steps: int):
        self.start_val = start_val
        self.end_val = end_val
        self.total_steps = total_steps

    def get_value(self, current_step: int) -> float:
        frac = min(current_step / max(1, self.total_steps), 1.0)
        return self.start_val + frac * (self.end_val - self.start_val)


_use_ent_sched = getattr(cfg, "use_entropy_scheduler", True)
if _use_ent_sched:
    entropy_scheduler = LinearScheduler(
        start_val=cfg.c_entropy,
        end_val=cfg.c_entropy_end,
        total_steps=cfg.max_iterations,
    )
else:
    entropy_scheduler = None

obs_np, _ = vec.reset(seed=None)
ep_returns: list[float] = []
ep_lengths: list[int] = []
ep_wins: list[bool] = []
running_rewards = np.zeros(cfg.num_envs)
running_lens = np.zeros(cfg.num_envs, dtype=int)

for iteration in range(start_iteration, cfg.max_iterations + 1):
    if entropy_scheduler is not None:
        cfg.c_entropy = entropy_scheduler.get_value(iteration)
    agent.eval()
    for t in range(cfg.rollout_steps):
        obs_t = dict_to_tensors(obs_np, device)
        card_mask = get_card_mask(obs_t)

        with torch.no_grad():
            sel_logits, exec_logits, value = agent(obs_t)

        sel_logits = mask_logits(sel_logits, card_mask)
        sel_dist = Categorical(logits=sel_logits)
        exec_dist = Categorical(logits=exec_logits)

        card_sels = sel_dist.sample()
        executions = exec_dist.sample()

        log_probs = (
            sel_dist.log_prob(card_sels).sum(dim=-1)
            + exec_dist.log_prob(executions)
        )

        actions_np = np.concatenate(
            [
                card_sels.cpu().numpy(),
                executions.cpu().numpy()[:, None],
            ],
            axis=-1,
        )

        next_obs_np, rewards_np, terms, truncs, infos = vec.step(actions_np)
        dones_np = np.logical_or(terms, truncs)
        gammas_np = _per_env_pbrs_gamma(infos, cfg.num_envs)

        buffer.store_step(
            t,
            obs_t,
            card_sels,
            executions,
            log_probs,
            value.squeeze(-1),
            rewards_np,
            dones_np,
            gammas_np,
        )

        running_rewards += rewards_np
        running_lens += 1
        ce_vec = _episode_wins(infos, cfg.num_envs, dones_np)
        for i in np.where(dones_np)[0]:
            ep_returns.append(float(running_rewards[i]))
            ep_lengths.append(int(running_lens[i]))
            ep_wins.append(bool(ce_vec[i]))
            running_rewards[i] = 0.0
            running_lens[i] = 0

        obs_np = next_obs_np

    with torch.no_grad():
        _, _, next_vals = agent(dict_to_tensors(obs_np, device))
        next_vals = next_vals.squeeze(-1)

    advantages, returns = compute_gae_vectorized(
        buffer.rewards,
        buffer.values,
        next_vals,
        buffer.dones,
        buffer.step_gammas,
        cfg.gae_lambda,
    )

    flat_obs, flat_card_sels, flat_execs, flat_lp, flat_vals = buffer.flatten()
    flat_adv = advantages.reshape(-1)
    flat_ret = returns.reshape(-1)

    agent.train()
    losses = ppo_update(
        agent,
        optimizer,
        flat_obs,
        flat_card_sels,
        flat_execs,
        flat_lp,
        flat_vals,
        flat_adv,
        flat_ret,
        cfg,
    )

    log_lr = float(optimizer.param_groups[0]["lr"])
    if lr_scheduler is not None:
        lr_scheduler.step()

    if iteration % CKPT_EVERY == 0:
        ckpt_dir = REPO_ROOT / "checkpoints"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        periodic_path = ckpt_dir / f"minimal_combat_ppo_iter_{iteration}.pt"
        torch.save(
            {
                "model_state_dict": agent.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": cfg,
                "iteration": iteration,
            },
            periodic_path,
        )
        print(
            f"Periodic checkpoint saved to {periodic_path}  "
            f"lr={log_lr:.2e}  c_entropy={cfg.c_entropy:.5f}  clip_eps={cfg.clip_eps:.4f}"
        )

    if iteration % 5 == 0 or iteration == 1:
        recent_r = np.mean(ep_returns[-50:]) if ep_returns else 0.0
        recent_len = np.mean(ep_lengths[-50:]) if ep_lengths else 0.0
        recent_win = 100.0 * np.mean(ep_wins[-50:]) if ep_wins else 0.0
        print(
            f"[iter {iteration:4d}]  "
            f"reward={recent_r:+.2f}  len={recent_len:.1f}  "
            f"win={recent_win:.1f}%  "
            f"pg={losses['pg_loss']:.4f}  "
            f"vf={losses['value_loss']:.4f}  "
            f"ent={losses['entropy']:.4f}  "
            f"episodes={len(ep_returns)}"
        )
        agent.eval()
        with torch.no_grad():
            log_obs_t = dict_to_tensors(obs_np, device)
            log_cm = get_card_mask(log_obs_t)
            log_sel_logits, _, _ = agent(log_obs_t)
            log_sel_logits = mask_logits(log_sel_logits, log_cm)
            p_sel = torch.softmax(log_sel_logits, dim=-1)[:, :, 1].cpu().numpy()
        n_slot = int(log_cm[0].sum().item())
        probs0 = [float(p_sel[0, i]) for i in range(n_slot)]
        p_std = float(np.std(probs0)) if probs0 else 0.0
        p_list = ", ".join(f"{p:.3f}" for p in probs0)
        # print(
        #     f"     P(select) env0 slots [0:{n_slot}]: [{p_list}]  std={p_std:.3f}  "
        #     f"lr={log_lr:.2e}  c_entropy={cfg.c_entropy:.5f}  clip_eps={cfg.clip_eps:.4f}"
        # )


[iter    1]  reward=+7.53  len=7.1  win=92.0%  pg=-0.0024  vf=7.7890  ent=2.3496  episodes=1123
[iter    5]  reward=+7.19  len=6.7  win=88.0%  pg=-0.0012  vf=11.5038  ent=2.2708  episodes=5753
[iter   10]  reward=+7.83  len=6.6  win=94.0%  pg=-0.0007  vf=8.7371  ent=2.3033  episodes=11543
[iter   15]  reward=+7.56  len=7.3  win=92.0%  pg=-0.0000  vf=9.8507  ent=2.2886  episodes=17298
[iter   20]  reward=+7.37  len=6.5  win=90.0%  pg=-0.0026  vf=7.6753  ent=2.2519  episodes=23172
[iter   25]  reward=+7.39  len=7.3  win=94.0%  pg=-0.0015  vf=8.7467  ent=2.2815  episodes=29063
[iter   30]  reward=+7.13  len=7.1  win=90.0%  pg=-0.0008  vf=8.7705  ent=2.2482  episodes=34926
[iter   35]  reward=+6.96  len=7.0  win=88.0%  pg=-0.0005  vf=15.3697  ent=2.2106  episodes=40713
[iter   40]  reward=+7.94  len=6.5  win=94.0%  pg=-0.0013  vf=8.0383  ent=2.3113  episodes=46542
[iter   45]  reward=+7.46  len=7.0  win=92.0%  pg=-0.0005  vf=8.7233  ent=2.2942  episodes=52390
[iter   50]  reward=+7.56  len

In [7]:
vec.close()
